In [1]:

# Install required libraries
!pip install -q gradio reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 17.6 MB/s eta 0:00:00


In [2]:

import gradio as gr
import re
import urllib.parse
from datetime import datetime
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet



## Database Schema


In [3]:

SCHEMA = {
    "users": ["id","name","email","signup_date","age","country","status"],
    "orders": ["id","user_id","product_name","amount","order_date","status"],
    "products": ["id","name","price","category","stock"],
    "transactions": ["id","user_id","amount","date","type"]
}



## Security Validation


In [4]:

DANGEROUS_KEYWORDS = ["DROP","DELETE","ALTER","TRUNCATE","INSERT","UPDATE"]

def is_safe(query):
    q = query.upper()
    for word in DANGEROUS_KEYWORDS:
        if word in q:
            return False
    if ";" in query or "--" in query:
        return False
    return True



## Detect Table


In [5]:

def detect_table(text):

    text = text.lower()

    mapping = {
        "users":["user","users","customer","account"],
        "orders":["order","orders","purchase","buy"],
        "products":["product","products","item"],
        "transactions":["transaction","payment","transfer"]
    }

    for table,words in mapping.items():
        for w in words:
            if w in text:
                return table

    return None



## Extract Conditions


In [6]:

def extract_conditions(text):

    text = text.lower()
    conditions = []

    if "last month" in text:
        conditions.append("DATE(signup_date)>=DATE('now','-1 month')")

    if "last week" in text:
        conditions.append("DATE(date)>=DATE('now','-7 days')")

    if "active" in text:
        conditions.append("status='active'")

    match = re.search(r'greater than (\d+)', text)

    if match:
        conditions.append(f"amount > {match.group(1)}")

    return conditions



## SQL Generator


In [7]:

query_history = []

def generate_sql(user_input):

    if not is_safe(user_input):
        return "⚠️ Security violation detected"

    table = detect_table(user_input)

    if table is None:
        return "❌ Table not detected"

    conditions = extract_conditions(user_input)

    sql = f"SELECT * FROM {table}"

    if conditions:
        sql += " WHERE " + " AND ".join(conditions)

    sql += " LIMIT 10"

    query_history.append(sql)

    return sql



## Export PDF


In [8]:

def export_pdf():

    filename = f"queries_{datetime.now().timestamp()}.pdf"

    styles = getSampleStyleSheet()

    story = []

    for q in query_history:

        story.append(Paragraph(q,styles["BodyText"]))
        story.append(Spacer(1,10))

    doc = SimpleDocTemplate(filename)
    doc.build(story)

    return filename



## Share via WhatsApp


In [9]:

def whatsapp_share():

    if len(query_history) == 0:
        return "No queries yet"

    msg = "\n".join(query_history[-5:])

    link = "https://wa.me/?text=" + urllib.parse.quote(msg)

    return link



## Chat Function


In [10]:

def chat(message, history):

    sql = generate_sql(message)

    history.append((message,sql))

    return history,""



## Launch UI


In [11]:

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🧠 SQL Query Generator")
    gr.Markdown("Convert Natural Language → SQL")

    chatbot = gr.Chatbot(height=400)

    msg = gr.Textbox(
        placeholder="Ask a database question..."
    )

    with gr.Row():
        send = gr.Button("Generate SQL")
        pdf_btn = gr.Button("Export PDF")
        wa_btn = gr.Button("Share WhatsApp")

    pdf_file = gr.File()
    share_box = gr.Textbox()

    send.click(chat,[msg,chatbot],[chatbot,msg])
    pdf_btn.click(export_pdf,outputs=pdf_file)
    wa_btn.click(whatsapp_share,outputs=share_box)

demo.launch(share=True)


/tmp/ipykernel_189/3181482908.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_189/3181482908.py:6: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)
/tmp/ipykernel_189/3181482908.py:6: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://40e58e7a6c6344bf84.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
